In [ ]:
import geopandas as gpd
from shapely.geometry import box
import sys

In [ ]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from seed_clustering import (
    find_seed_centroids_single
)
from cluster_construction import (
    find_and_filter_nearby_buildings,
    create_cluster_subgraph,
)
from distance_metrics import (
    road_network_distance_to_centroid,
)
from load_profile_processing import (
    get_building_load_profiles,
)
from dinkelbach_milp import (
    solve_dinkelbach_milp
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints,
    join_ornl_overture_buildings
    
)
from osmnx_api import (
    get_road_network,
)

In [ ]:
from plotting_functions import (
    plot_timeseries_loads,
    plot_pareto_fontier
)

# Download Building and Road Network Data

In [ ]:
# Get the bounding box of the area that is to be investigated
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [ ]:
# Get the pre-queried GHE / waste heat source locations
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")

# Crop the locations to the bounding box
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())

In [ ]:
# Get building data from Overture Maps and ORNL USA Structures 
overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)

# Crop the buildings to the bounding box
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)
overture_buildings = overture_buildings.to_crs(cropped_ghes.crs)

In [ ]:
# Join the building data together
buildings_joined = join_ornl_overture_buildings(
    ornl_data=ornl_buildings,
    overture_data=overture_buildings
)

In [ ]:
# Get road network data
# Ensure that it is a non-directed graph
network = get_road_network(bbox=TEST_BBOX)
network = network.to_undirected()
network.is_directed()

# Get Cluster Seeds and Nearby Buildings 

In [ ]:
# Create seed centroids from each GHE / waste heat source location
cropped_ghes, seed_centroids = find_seed_centroids_single(
    ghe_locations=cropped_ghes,
)

# For each of the centroid
# gather all buildings within a certain road network radiuss
nearby_buildings = find_and_filter_nearby_buildings(
    seed_centroids=seed_centroids,
    building_data=buildings_joined,
    G=network,
    max_distance=500,
)

In [ ]:
cluster_idx = 250

# Get all buildings near the selected cluster 
# and the centroid associated with that cluster 
current_cluster = nearby_buildings[nearby_buildings['cluster'] == cluster_idx].copy()
cluster_seed_centroid = seed_centroids.loc[seed_centroids['cluster'] == cluster_idx, 'geometry'].item()

# Assign a distance to each building in the cluster 
current_cluster['centroid_distance'] = road_network_distance_to_centroid(
    buildings_df=current_cluster,
    centroid=cluster_seed_centroid,
    G=network,
    max_distance=500
)

# Download Representative Load Profile Curves

In [ ]:
# For each of the buildings, gather each of the necessary data
building_types = current_cluster['comstock_type'].tolist()
building_sizes = current_cluster['sqfeet'].tolist()
building_distances = current_cluster['centroid_distance'].tolist()
building_sources = [
    'restock' if building_type == "Multi-Family with 5+ Units" else 'comstock'
    for building_type in building_types
]

In [ ]:
# Gather the load profiles for each of the buildings in the selected cluster
load_profile_frames = get_building_load_profiles(
    building_types=building_types,
    building_sizes=building_sizes,
    building_sources=building_sources,
    state='WA',
    downsample=True # Downsample to daily time frequency
)

# Get the loads for each of the buildings 
building_loads = [building['net_load'].tolist() for building in load_profile_frames]

In [ ]:
plot_pareto_fontier(
    building_loads=building_loads[:15],
    building_sizes=building_sizes[:15],
    seed_idx=0,
    k_min=5
)